# Multiprocessor Ronit IDK Cascade (REAL TIME CLASSIFICATION)

This notebook keeps the same RF IDK routing as `sequential_ronit_IDK_cascades.ipynb`, but runs ResNet-18, ResNet-34, and ResNet-50 in separate MPS worker processes.


This cell imports the packages and finds the repo paths used by the notebook.


In [1]:
from collections import Counter, deque
from pathlib import Path
import sys
import tarfile
import time

import numpy as np
import torch
import torch.multiprocessing as mp
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from torchvision import transforms

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from real_time_mp_workers import model_worker


This cell sets the models, router training caches, test dataset, and thresholds.


In [2]:
MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet152"
MODELS = (MODEL_A, MODEL_B, MODEL_C)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

TRAIN_CACHE_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
TEST_SAMPLES = 10000

CLASSIFICATION_THRESHOLD = 0.7
SKIP = 0
PREDICT = 1
MAX_IN_FLIGHT_BY_MODEL = {model_name: 1 for model_name in MODELS}

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this multiprocessor notebook")

DEVICE_BY_MODEL = {model_name: "mps" for model_name in MODELS}


This cell streams images and labels directly from the local ImageNet-V2 tar files.


In [3]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break


This cell prepares ImageNet preprocessing and starts one MPS process per ResNet model.


In [4]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def image_to_batch(image):
    return preprocess(image).unsqueeze(0)


def start_model_processes():
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.SimpleQueue() for model_name in MODELS}
    result_queue = ctx.SimpleQueue()
    processes = []

    for model_name in MODELS:
        process = ctx.Process(
            target=model_worker,
            args=(model_name, DEVICE_BY_MODEL[model_name], job_queues[model_name], result_queue),
        )
        process.start()
        processes.append(process)

    return job_queues, result_queue, processes


def stop_model_processes(job_queues, processes):
    for queue in job_queues.values():
        queue.put(None)
    for process in processes:
        process.join()


This cell defines the Random Forest feature helper used by the router.


In [5]:
def probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)


This cell builds Random Forest training rows from the saved ResNet-18 and ResNet-34 artifact caches.


In [6]:
def load_cache(prefix, model_name):
    with np.load(ARTIFACTS_DIR / f"{prefix}_{model_name}.npz") as data:
        return {"probabilities": data["probabilities"]}


def build_training_data():
    feature_parts = []
    label_parts = []

    for prefix in TRAIN_CACHE_PREFIXES:
        a_cache = load_cache(prefix, MODEL_A)
        b_cache = load_cache(prefix, MODEL_B)
        a_probs = a_cache["probabilities"]
        a_idk = a_probs.max(axis=1) < CLASSIFICATION_THRESHOLD

        X = probability_features(a_probs[a_idk])
        y = np.where(
            b_cache["probabilities"].max(axis=1)[a_idk] < CLASSIFICATION_THRESHOLD,
            SKIP,
            PREDICT,
        ).astype(np.int64)

        feature_parts.append(X)
        label_parts.append(y)
        print(prefix, "training rows:", len(y))

    X = np.concatenate(feature_parts)
    y = np.concatenate(label_parts)
    print("Training rows:", len(y), "skip:", int((y == SKIP).sum()), "predict:", int((y == PREDICT).sum()))
    return X, y


train_X, train_y = build_training_data()


matched training rows: 5205
top training rows: 3834
Training rows: 9039 skip: 6373 predict: 2666


This cell trains the same Random Forest router as the sequential Ronit cascade.


In [7]:
router_name = "random_forest"
router = RandomForestClassifier(
    n_estimators=50,
    max_depth=4,
    min_samples_leaf=40,
    class_weight="balanced",
    random_state=42,
    n_jobs=1,
)
router.fit(train_X, train_y)
print("Router:", router_name)


Router: random_forest


This cell names the cascade and prints worker placement.


In [8]:
cascade_name = "multiprocessor_ronit_idk"

print("Cascade:", cascade_name)
print("Models:", MODELS)
print("Device by model:", DEVICE_BY_MODEL)
print("Max in flight by model:", MAX_IN_FLIGHT_BY_MODEL)


Cascade: multiprocessor_ronit_idk
Models: ('resnet18', 'resnet34', 'resnet152')
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet152': 'mps'}
Max in flight by model: {'resnet18': 1, 'resnet34': 1, 'resnet152': 1}


This cell runs the RF-routed cascade as a three-worker pipeline.


In [9]:
def run_multiprocessor_ronit_idk_cascade():
    job_queues, result_queue, processes = start_model_processes()

    try:
        labels = np.full(TEST_SAMPLES, -1, dtype=np.int64)
        final_predictions = np.full(TEST_SAMPLES, -1, dtype=np.int64)
        final_models = np.full(TEST_SAMPLES, "", dtype="<U32")
        latencies_ms = np.full(TEST_SAMPLES, np.nan, dtype=np.float64)

        sample_tensors = {}
        sample_start_times = {}
        pending_by_model = {model_name: deque() for model_name in MODELS}
        in_flight_by_model = Counter()
        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        final_count_by_model = Counter()
        idk_count_by_model = Counter()
        route_count_to_model = Counter()
        queue_max_size_by_model = Counter()
        router_call_count = 0
        router_time_ms_total = 0.0

        row_iterator = iter(stream_rows(TEST_VARIANT, TEST_SAMPLES))
        next_sample_index = 0
        completed_sample_count = 0
        input_exhausted = False
        run_start = time.perf_counter()

        def enqueue_model(model_name, sample_index):
            pending_by_model[model_name].append(sample_index)
            queue_max_size_by_model[model_name] = max(
                queue_max_size_by_model[model_name],
                len(pending_by_model[model_name]),
            )

        def dispatch_model(model_name):
            while pending_by_model[model_name] and in_flight_by_model[model_name] < MAX_IN_FLIGHT_BY_MODEL[model_name]:
                sample_index = pending_by_model[model_name].popleft()
                job_queues[model_name].put((sample_index, sample_tensors[sample_index]))
                in_flight_by_model[model_name] += 1
                execution_count_by_model[model_name] += 1

        def dispatch_all_models():
            for model_name in MODELS:
                dispatch_model(model_name)

        def load_next_inputs():
            nonlocal input_exhausted, next_sample_index
            while (
                not input_exhausted
                and next_sample_index < TEST_SAMPLES
                and len(pending_by_model[MODEL_A]) + in_flight_by_model[MODEL_A] < MAX_IN_FLIGHT_BY_MODEL[MODEL_A]
            ):
                try:
                    image, label = next(row_iterator)
                except StopIteration:
                    input_exhausted = True
                    break

                sample_index = next_sample_index
                next_sample_index += 1
                labels[sample_index] = int(label)
                sample_tensors[sample_index] = image_to_batch(image)
                sample_start_times[sample_index] = time.perf_counter()
                enqueue_model(MODEL_A, sample_index)

            if next_sample_index >= TEST_SAMPLES:
                input_exhausted = True

        def finish_sample(sample_index, model_name, prediction):
            nonlocal completed_sample_count
            final_predictions[sample_index] = prediction
            final_models[sample_index] = model_name
            final_count_by_model[model_name] += 1
            latencies_ms[sample_index] = (time.perf_counter() - sample_start_times.pop(sample_index)) * 1000.0
            sample_tensors.pop(sample_index, None)
            completed_sample_count += 1

        def route_from_resnet18(sample_index, probabilities):
            nonlocal router_call_count, router_time_ms_total
            router_start = time.perf_counter()
            route = int(router.predict(probability_features(probabilities[None, :]))[0])
            router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
            router_call_count += 1
            next_model = MODEL_C if route == SKIP else MODEL_B
            route_count_to_model[next_model] += 1
            enqueue_model(next_model, sample_index)

        while not input_exhausted or completed_sample_count < next_sample_index:
            load_next_inputs()
            dispatch_all_models()

            if sum(in_flight_by_model.values()) == 0:
                if input_exhausted:
                    break
                continue

            message = result_queue.get()
            if message[0] == "error":
                _, worker_name, error = message
                raise RuntimeError(f"{worker_name} worker failed: {error}")

            sample_index, model_name, probabilities, prediction, confidence, elapsed_ms = message
            in_flight_by_model[model_name] -= 1
            execution_time_ms_by_model[model_name] += elapsed_ms

            if confidence < CLASSIFICATION_THRESHOLD:
                idk_count_by_model[model_name] += 1

            if model_name == MODEL_A:
                if confidence >= CLASSIFICATION_THRESHOLD:
                    finish_sample(sample_index, MODEL_A, prediction)
                else:
                    route_from_resnet18(sample_index, probabilities)
            elif model_name == MODEL_B:
                if confidence >= CLASSIFICATION_THRESHOLD:
                    finish_sample(sample_index, MODEL_B, prediction)
                else:
                    route_count_to_model[MODEL_C] += 1
                    enqueue_model(MODEL_C, sample_index)
            else:
                finish_sample(sample_index, MODEL_C, prediction)

            dispatch_all_models()

        labels = labels[:next_sample_index]
        final_predictions = final_predictions[:next_sample_index]
        final_models = final_models[:next_sample_index]
        latencies_ms = latencies_ms[:next_sample_index]
        total_wall_time_seconds = time.perf_counter() - run_start
        correct_predictions = int(np.count_nonzero(final_predictions == labels))

        return {
            "total_samples": int(len(labels)),
            "accuracy": float(correct_predictions / len(labels)),
            "correct_predictions": correct_predictions,
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "throughput_fps": float(len(labels) / total_wall_time_seconds),
            "mean_latency_ms": float(latencies_ms.mean()),
            "cascade_name": cascade_name,
            "router_name": router_name,
            "router_call_count": int(router_call_count),
            "total_router_time_ms": float(router_time_ms_total),
            "mean_router_time_ms": float(router_time_ms_total / router_call_count) if router_call_count else 0.0,
            "device_by_model": DEVICE_BY_MODEL,
            "max_in_flight_by_model": MAX_IN_FLIGHT_BY_MODEL,
            "final_prediction_count_by_model": {model: int(final_count_by_model[model]) for model in MODELS},
            "execution_count_by_model": {model: int(execution_count_by_model[model]) for model in MODELS},
            "mean_execution_time_ms_by_model": {
                model: float(execution_time_ms_by_model[model] / execution_count_by_model[model])
                if execution_count_by_model[model]
                else 0.0
                for model in MODELS
            },
            "idle_time_ms_by_model": {
                model: max(0.0, total_wall_time_seconds * 1000.0 - execution_time_ms_by_model[model])
                for model in MODELS
            },
            "idk_count_by_model": {model: int(idk_count_by_model[model]) for model in MODELS},
            "route_count_to_model": {model: int(route_count_to_model[model]) for model in MODELS},
            "queue_max_size_by_model": {model: int(queue_max_size_by_model[model]) for model in MODELS},
        }

    finally:
        stop_model_processes(job_queues, processes)


results = run_multiprocessor_ronit_idk_cascade()


This cell prints the relevant metrics for the multiprocessor Ronit run.


In [10]:
print("Real-Time Multiprocessor Ronit MPS Test")
print("Confidence Threshold Constant: ", CLASSIFICATION_THRESHOLD)
print("Cascade:", results["cascade_name"])
print("Device by model:", results["device_by_model"])
print("Max in flight by model:", results["max_in_flight_by_model"])
print("Total samples:", results["total_samples"])
print("Accuracy:", round(results["accuracy"], 4))
print("Correct predictions:", results["correct_predictions"])
print("Total wall time (seconds):", round(results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(results["throughput_fps"], 3))
print("Mean latency (ms):", round(results["mean_latency_ms"], 3))
print()
print(f"Classifier {results['router_name']} router:")
print(f"  Router calls: {results['router_call_count']}")
print(f"  Total router time (ms): {results['total_router_time_ms']:.3f}")
print(f"  Mean router time (ms): {results['mean_router_time_ms']:.6f}")
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['execution_count_by_model'][model_name]}")
print("IDK count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['idk_count_by_model'][model_name]}")
print("Route count to model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['route_count_to_model'][model_name]}")
print("Max pending queue size by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['queue_max_size_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {results['mean_execution_time_ms_by_model'][model_name]:.3f}")
print("Idle time by model (seconds):")
for model_name in MODELS:
    print(f"  {model_name}: {results['idle_time_ms_by_model'][model_name] / 1000.0:.3f}")


Real-Time Multiprocessor Ronit MPS Test
Confidence Threshold Constant:  0.7
Cascade: multiprocessor_ronit_idk
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet152': 'mps'}
Max in flight by model: {'resnet18': 1, 'resnet34': 1, 'resnet152': 1}
Total samples: 10000
Accuracy: 0.7718
Correct predictions: 7718
Total wall time (seconds): 243.021
Throughput (FPS): 41.149
Mean latency (ms): 3339.886

Classifier random_forest router:
  Router calls: 4445
  Total router time (ms): 12042.292
  Mean router time (ms): 2.709177

Final prediction count by model:
  resnet18: 5555
  resnet34: 986
  resnet152: 3459
Execution count by model:
  resnet18: 10000
  resnet34: 2293
  resnet152: 3459
IDK count by model:
  resnet18: 4445
  resnet34: 1307
  resnet152: 2449
Route count to model:
  resnet18: 0
  resnet34: 2293
  resnet152: 3459
Max pending queue size by model:
  resnet18: 1
  resnet34: 1
  resnet152: 287
Mean execution time by model (ms):
  resnet18: 17.448
  resnet34: 20.121
  resnet